In [1]:
import numpy as np
import pandas as pd

In [5]:
# Load the dataset
data = pd.read_csv("Vahan_Case_Study.csv")
data.head()

,upload_date,lead_source,candidate_phone,Uploaded Leads,Attempted,Connected,Attempt per Lead,tag_filled,Interested,OB_after_upload,OB_after_first_attempt,FT_after_upload,FT_after_first_attempt,upload_to_first_attempt_P50 (hrs),Attempted %,Attempt to Connected %,Connect to Interested %,Interested to FT_after_first_attempt %,Attempted to FT_after_upload %
0,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,91148e89d497914604c9c248ccd2aca212fac665bdc396...,1,1,1,1.0,1,0,0,0,0,0,112.0,100,100.0,0.0,NaN,0.0
1,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,4aa4e7a4f4a3b9fa608c4c1729be6a8accc31f58b3d8b1...,1,1,0,1.0,0,0,0,0,0,0,112.0,100,0.0,NaN,NaN,0.0
2,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,0e97e34090259c1ce6cef1fd1c491fcfc469ba9d2d1ce7...,1,1,0,1.0,0,0,0,0,0,0,89.0,100,0.0,NaN,NaN,0.0
3,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,cdc003718802cffa769a478bcc0fe31aa20ac3ffe97a7c...,1,1,0,1.0,0,0,0,0,0,0,106.0,100,0.0,NaN,NaN,0.0
4,18-07-2026,2W3W - 3WAHD - Khanna - 3W - 17 Jul,d70833a770ab63e1992254dad148b100bbbadbe0e94673...,1,1,0,1.0,0,0,0,0,0,0,90.0,100,0.0,NaN,NaN,0.0


## Data integrity checks (before any aggregation)

Run before trusting any cohort numbers:
1. **Duplicate leads across cohorts** — does the same `candidate_phone` show up under more than one `lead_source`? If so, summing `FT_after_upload` per cohort can double-count a single converted lead.
2. **Funnel monotonicity** — flag rows where a later funnel stage is marked `1` while an earlier one is `0` (e.g. `FT_after_upload=1` but `Connected=0`), since these are internally inconsistent and may not be real conversions.

In [ ]:
dupe_counts = data['candidate_phone'].value_counts()
dupe_phones = dupe_counts[dupe_counts > 1].index
dupe_rows = data[data['candidate_phone'].isin(dupe_phones)]
span = dupe_rows.groupby('candidate_phone')['lead_source'].nunique()
ft_per_phone = dupe_rows.groupby('candidate_phone')['FT_after_upload'].sum()

print(f"Repeated phone hashes: {len(dupe_phones)} ({len(dupe_rows)} rows, {len(dupe_rows)/len(data):.1%} of data)")
print(f"  -> all of them span multiple lead_source values: {(span > 1).all()}")
print(f"  -> phones double-credited with FT_after_upload=1 more than once: {(ft_per_phone > 1).sum()}")

print()
anomaly = (data['FT_after_upload'] == 1) & (data['Connected'] == 0)
print(f"FT_after_upload=1 rows with Connected=0 (internally inconsistent): {anomaly.sum()} of {data['FT_after_upload'].sum()} total FT events")
print(data.loc[anomaly, 'lead_source'].value_counts())

print()
# Fix 1: dedup to one row per lead, keeping their most-advanced funnel state,
# so a lead counted under two lead_source values can't inflate either cohort's FT count.
priority_cols = ['FT_after_upload', 'Interested', 'Connected', 'Attempted']
data_clean = (
    data.sort_values(priority_cols, ascending=False)
        .drop_duplicates(subset='candidate_phone', keep='first')
        .copy()
)
data_clean['ft_anomaly'] = (data_clean['FT_after_upload'] == 1) & (data_clean['Connected'] == 0)
print(f"Rows: {len(data)} -> {len(data_clean)} after dedup. FT_after_upload sum: {data['FT_after_upload'].sum()} -> {data_clean['FT_after_upload'].sum()}")

## Cohort analysis: which `lead_source` cohorts convert best?

`FT` = **First Trip** (confirmed). North-star metric: `FT_after_upload` rate — leads that make it all the way
to a first trip, not intermediate funnel steps like connect/interest rate.

Built on `data_clean` (deduped leads, one row per unique `candidate_phone`) so a lead appearing under two
`lead_source` values can't inflate either cohort's count. Cohorts under `MIN_LEADS` are excluded, "warm"
cohorts (already pre-qualified — fees already paid, or AI-prescreened) are ranked separately from "cold"
acquisition cohorts, and every rate is reported with a 95% Wilson confidence interval — with FT counts this
small (single digits per cohort), point estimates alone are misleading; cohorts whose intervals overlap are
**not** reliably distinguishable from each other and shouldn't be read as a clean ranking.

In [ ]:
def wilson_ci(successes, n, z=1.96):
    """95% Wilson score interval for a proportion, safe for small n / rare events."""
    if n == 0:
        return 0.0, 0.0, 0.0
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z * ((p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5)) / denom
    return p * 100, max(0, (center - half) * 100), (center + half) * 100

cohort = data_clean.groupby('lead_source').agg(
    leads=('Uploaded Leads', 'sum'),
    attempted=('Attempted', 'sum'),
    connected=('Connected', 'sum'),
    interested=('Interested', 'sum'),
    ft_after_upload=('FT_after_upload', 'sum'),
    ft_anomalies=('ft_anomaly', 'sum'),  # FT events with no logged Connected=1 -- data-quality flag, not dropped by default
).reset_index()

cohort['attempt_rate']  = cohort['attempted'] / cohort['leads'] * 100
cohort['connect_rate']  = cohort['connected'] / cohort['attempted'] * 100
cohort['interest_rate'] = cohort['interested'] / cohort['connected'] * 100

ci = cohort.apply(lambda r: wilson_ci(r['ft_after_upload'], r['leads']), axis=1, result_type='expand')
cohort[['ft_rate', 'ft_rate_ci_low', 'ft_rate_ci_high']] = ci

cohort.sort_values('ft_rate', ascending=False)

In [ ]:
MIN_LEADS = 100  # drop cohorts too small to draw a conclusion from (e.g. n=1, n=2)

# "Warm" cohorts are already pre-qualified (fees paid / AI pre-screened) rather than
# cold acquisition traffic, so they aren't comparable to the rest on funnel-rate metrics.
warm_keywords = ['PreOb', 'AI Connected']
cohort['segment'] = cohort['lead_source'].apply(
    lambda s: 'warm' if any(k in s for k in warm_keywords) else 'cold'
)

ranked = cohort[cohort['leads'] >= MIN_LEADS].sort_values('ft_rate', ascending=False)
cols = ['lead_source', 'segment', 'leads', 'ft_after_upload', 'ft_anomalies',
        'ft_rate', 'ft_rate_ci_low', 'ft_rate_ci_high']
ranked[cols]

In [ ]:
def report_top3(seg):
    sub = ranked[ranked['segment'] == seg].head(3).reset_index(drop=True)
    print(f"Top 3 {seg}-lead cohorts by FT rate (leads -> First Trip):")
    display(sub[cols])
    # Flag statistical ties: does each cohort's CI overlap the next one's?
    for i in range(len(sub) - 1):
        a, b = sub.iloc[i], sub.iloc[i + 1]
        if a['ft_rate_ci_low'] <= b['ft_rate_ci_high']:
            print(f"  NOTE: '{a['lead_source']}' (#{i+1}) and '{b['lead_source']}' (#{i+2}) "
                  f"have overlapping 95% CIs -- not reliably distinguishable given sample size.")
    if (sub['ft_anomalies'] > 0).any():
        print(f"  CAUTION: some of these cohorts' FT counts include FT_after_upload=1 rows with "
              f"Connected=0 (data-quality anomaly) -- see 'ft_anomalies' column.")
    print()

report_top3('cold')
report_top3('warm')

## SQL version (raw data as a table)

Treating `data` as a raw table `leads`, aggregated to the **`lead_source` (cohort) grain** — same level as the
pandas analysis above. `FT` = **First Trip**. The query:
1. Dedups to one row per `candidate_phone` (their most-advanced funnel state) via `ROW_NUMBER()`, so a lead
   appearing under multiple `lead_source` values can't inflate more than one cohort's FT count.
2. Tags each cohort `warm`/`cold` and drops cohorts under 100 leads.
3. Uses `RANK() OVER (PARTITION BY segment ORDER BY ft_rate DESC)` so "top 3" comes out of the query directly.

Wilson confidence intervals are computed afterward in pandas (simpler than in raw SQL) — see the next cell.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
data.to_sql('leads', conn, index=False, if_exists='replace')

query = """
WITH deduped AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY candidate_phone
               ORDER BY FT_after_upload DESC, Interested DESC, Connected DESC, Attempted DESC
           ) AS rn
    FROM leads
),
one_row_per_lead AS (
    SELECT * FROM deduped WHERE rn = 1
),
agg AS (
    SELECT
        lead_source,
        CASE
            WHEN lead_source LIKE '%PreOb%' OR lead_source LIKE '%AI Connected%' THEN 'warm'
            ELSE 'cold'
        END AS segment,
        SUM("Uploaded Leads")                                             AS leads,
        SUM(Attempted)                                                    AS attempted,
        SUM(Connected)                                                    AS connected,
        SUM(Interested)                                                   AS interested,
        SUM(FT_after_upload)                                              AS ft_after_upload,
        SUM(CASE WHEN FT_after_upload = 1 AND Connected = 0 THEN 1 ELSE 0 END) AS ft_anomalies,
        ROUND(100.0 * SUM(Attempted)       / SUM("Uploaded Leads"), 1)    AS attempt_rate,
        ROUND(100.0 * SUM(Connected)       / NULLIF(SUM(Attempted), 0), 1)  AS connect_rate,
        ROUND(100.0 * SUM(Interested)      / NULLIF(SUM(Connected), 0), 1) AS interest_rate,
        ROUND(100.0 * SUM(FT_after_upload) / SUM("Uploaded Leads"), 3)    AS ft_rate
    FROM one_row_per_lead
    GROUP BY lead_source
    HAVING SUM("Uploaded Leads") >= 100
)
SELECT *,
       RANK() OVER (PARTITION BY segment ORDER BY ft_rate DESC) AS rank_in_segment
FROM agg
ORDER BY segment, rank_in_segment
"""

agg = pd.read_sql_query(query, conn)
agg[['ft_rate_ci_low', 'ft_rate_ci_high']] = agg.apply(
    lambda r: pd.Series(wilson_ci(r['ft_after_upload'], r['leads'])[1:]), axis=1
)
agg

In [ ]:
# Top 3 cohorts per segment, straight from the query's rank_in_segment column
agg[agg['rank_in_segment'] <= 3].sort_values(['segment', 'rank_in_segment'])